# Fetal Health Classification using Cardiotocography Data

## Data Preprocessing

This notebook reimplements the preprocessing stage described in the paper, "Multiclass Classification of Fetal Health using Cardiotocogram Data."

The Cardiotocography dataset contains 21 features extracted from fetal heart rate and uterine contraction measurements. The objective is to classify fetal health into three categories:

- 1 — Normal
- 2 — Suspect
- 3 — Pathological

In [1]:
!pip install -q ucimlrepo

In [2]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

## Dataset Loading

The Cardiotocography dataset was obtained from the UCI Machine Learning Repository. It contains 2,126 observations and 21 input features.

The dataset also contains two target variables, `CLASS` and `NSP`. Since the reference paper performs three-class fetal health classification, `NSP` is used as the target variable.

In [3]:
# UCI dataset ID 193 = Cardiotocography
cardiotocography = fetch_ucirepo(id=193)

# 21 input features
X = cardiotocography.data.features.copy()

# Targets supplied by UCI
y_all = cardiotocography.data.targets.copy()

print("Features shape:", X.shape)
print("Target columns:", y_all.columns.tolist())

X.head()

Features shape: (2126, 21)
Target columns: ['CLASS', 'NSP']


,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency
0,120,0.000,0.0,0.000,0.000,0.0,0.0,73,0.5,43,...,64,62,126,2,0,120,137,121,73,1
1,132,0.006,0.0,0.006,0.003,0.0,0.0,17,2.1,0,...,130,68,198,6,1,141,136,140,12,0
2,133,0.003,0.0,0.008,0.003,0.0,0.0,16,2.1,0,...,130,68,198,5,1,141,135,138,13,0
3,134,0.003,0.0,0.008,0.003,0.0,0.0,16,2.4,0,...,117,53,170,11,0,137,134,137,13,1
4,132,0.007,0.0,0.008,0.000,0.0,0.0,16,2.4,0,...,117,53,170,9,0,137,136,138,11,1


## Target Variable

The `NSP` variable represents fetal health status:

- 1 — Normal
- 2 — Suspect
- 3 — Pathological

This variable is used as the target for the multiclass classification problem.

In [4]:
y = y_all["NSP"].copy()

print(y.value_counts().sort_index())

NSP
1    1655
2     295
3     176
Name: count, dtype: int64


## Feature Preparation

The original abbreviated feature names were renamed to more descriptive names to improve readability and make the dataset easier to interpret during analysis.

In [5]:
rename_dict = {
    "LB": "baseline value",
    "AC": "accelerations",
    "FM": "fetal_movement",
    "UC": "uterine_contractions",
    "DL": "light_decelerations",
    "DS": "severe_decelerations",
    "DP": "prolongued_decelerations",
    "ASTV": "abnormal_short_term_variability",
    "MSTV": "mean_value_of_short_term_variability",
    "ALTV": "percentage_of_time_with_abnormal_long_term_variability",
    "MLTV": "mean_value_of_long_term_variability",
    "Width": "histogram_width",
    "Min": "histogram_min",
    "Max": "histogram_max",
    "Nmax": "histogram_number_of_peaks",
    "Nzeros": "histogram_number_of_zeroes",
    "Mode": "histogram_mode",
    "Mean": "histogram_mean",
    "Median": "histogram_median",
    "Variance": "histogram_variance",
    "Tendency": "histogram_tendency"
}

X = X.rename(columns=rename_dict)

print(X.columns.tolist())

['baseline value', 'accelerations', 'fetal_movement', 'uterine_contractions', 'light_decelerations', 'severe_decelerations', 'prolongued_decelerations', 'abnormal_short_term_variability', 'mean_value_of_short_term_variability', 'percentage_of_time_with_abnormal_long_term_variability', 'mean_value_of_long_term_variability', 'histogram_width', 'histogram_min', 'histogram_max', 'histogram_number_of_peaks', 'histogram_number_of_zeroes', 'histogram_mode', 'histogram_mean', 'histogram_median', 'histogram_variance', 'histogram_tendency']


In [6]:
df = X.copy()
df["fetal_health"] = y.values

print("Original shape:", df.shape)
df.head()

Original shape: (2126, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120,0.000,0.0,0.000,0.000,0.0,0.0,73,0.5,43,...,62,126,2,0,120,137,121,73,1,2
1,132,0.006,0.0,0.006,0.003,0.0,0.0,17,2.1,0,...,68,198,6,1,141,136,140,12,0,1
2,133,0.003,0.0,0.008,0.003,0.0,0.0,16,2.1,0,...,68,198,5,1,141,135,138,13,0,1
3,134,0.003,0.0,0.008,0.003,0.0,0.0,16,2.4,0,...,53,170,11,0,137,134,137,13,1,1
4,132,0.007,0.0,0.008,0.000,0.0,0.0,16,2.4,0,...,53,170,9,0,137,136,138,11,1,1


## Data Cleaning

The dataset was inspected for missing values and duplicate observations.

Following the preprocessing methodology described in the paper:

1. Missing values were checked and removed if present.
2. Duplicate observations were identified.
3. Duplicate rows were removed before further analysis.

In [7]:
missing_values = df.isnull().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())

Missing values per column:
baseline value                                            0
accelerations                                             0
fetal_movement                                            0
uterine_contractions                                      0
light_decelerations                                       0
severe_decelerations                                      0
prolongued_decelerations                                  0
abnormal_short_term_variability                           0
mean_value_of_short_term_variability                      0
percentage_of_time_with_abnormal_long_term_variability    0
mean_value_of_long_term_variability                       0
histogram_width                                           0
histogram_min                                             0
histogram_max                                             0
histogram_number_of_peaks                                 0
histogram_number_of_zeroes                                0
histogram_mod

In [8]:
rows_before = len(df)

df = df.dropna().copy()

rows_after = len(df)

print("Rows before removing nulls:", rows_before)
print("Rows after removing nulls :", rows_after)
print("Rows removed              :", rows_before - rows_after)

Rows before removing nulls: 2126
Rows after removing nulls : 2126
Rows removed              : 0


In [9]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 13


In [10]:
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (2113, 22)


In [11]:
class_counts = df["fetal_health"].value_counts().sort_index()

print(class_counts)

fetal_health
1    1646
2     292
3     175
Name: count, dtype: int64


### Data Cleaning Results

The original dataset contained 2,126 observations.

No missing observations were identified. A total of 13 duplicate observations were detected and removed, reducing the dataset to 2,113 unique samples.

The final class distribution was:

- Normal: 1,646 samples
- Suspect: 292 samples
- Pathological: 175 samples

These values match the preprocessing results reported in the paper.

## Target Label Mapping

For easier interpretation and visualization, the numerical fetal health classes were mapped to descriptive labels:

- 1 → Normal
- 2 → Suspect
- 3 → Pathological

The original numerical target is retained for machine learning.

In [12]:
health_labels = {
    1: "Normal",
    2: "Suspect",
    3: "Pathological"
}

df["fetal_health_label"] = df["fetal_health"].map(health_labels)

df[["fetal_health", "fetal_health_label"]].head()

,fetal_health,fetal_health_label
0,2,Suspect
1,1,Normal
2,1,Normal
3,1,Normal
4,1,Normal


## Final Data Verification

Before saving the cleaned dataset, the final dataset was verified to ensure that:

- 2,113 observations remained
- 21 input features were present
- No missing values remained
- No duplicate observations remained
- The expected fetal health class distribution was preserved

In [17]:
print("-" * 50)
print("FINAL CLEANED DATASET")
print("-" * 50)

print("Rows:", len(df))
print("Input features:", 21)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.drop(columns=["fetal_health_label"]).duplicated().sum())

print("\nClass distribution:")
print(df["fetal_health_label"].value_counts())

print("\nDataset shape:")
print(df.shape)

--------------------------------------------------
FINAL CLEANED DATASET
--------------------------------------------------
Rows: 2113
Input features: 21
Missing values: 0
Duplicate rows: 0

Class distribution:
fetal_health_label
Normal          1646
Suspect          292
Pathological     175
Name: count, dtype: int64

Dataset shape:
(2113, 23)


## Export Cleaned Dataset

The descriptive text label is removed before export so that the final dataset contains the original 21 numerical input features and one numerical target variable.

The cleaned dataset is saved as `fetal_health_cleaned.csv` for use in exploratory data analysis, feature selection, and machine learning model training.

In [14]:
df_clean = df.drop(columns=["fetal_health_label"])

df_clean.to_csv("fetal_health_cleaned.csv", index=False)

print("Saved fetal_health_cleaned.csv")
print(df_clean.shape)

Saved fetal_health_cleaned.csv
(2113, 22)


In [16]:
from google.colab import files

files.download("fetal_health_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>